In [1]:
import pandas as pd, numpy as np
from scipy import stats
from itertools import permutations
p='/workspace/t3-prism-bo-batch-drop-results.csv'
df=pd.read_csv(p)
m=df.dropna(subset=['mass_g','spec']).copy()
m['E_mJ']=m.e_rebound_mean*m.mass_g*9.80665*1.524
m['cv_pct']=100*m.t180_sd/m.t180_mean
print('mapped n',len(m))
print(m[['specimen','spec','mass_g','t180_mean','t180_sd','cv_pct','E_mJ']].to_string(index=False))
# Pearson mass
r,pv=stats.pearsonr(m.mass_g,m.t180_mean)
z=np.arctanh(r); se=1/np.sqrt(len(m)-3); ci=np.tanh(z+np.array([-1,1])*stats.norm.ppf(.975)*se)
print('\nmass Pearson',r,pv,'Fisher95',ci)
# leave-one-out mass correlations
print('mass r leave-one-out')
for i,row in m.iterrows():
 x=m.drop(i); print(row.specimen, stats.pearsonr(x.mass_g,x.t180_mean).statistic)
# Spearman tradeoff exact permutation two-sided based abs rho
rho=stats.spearmanr(m.t180_mean,m.E_mJ).statistic
ys=stats.rankdata(m.E_mJ); xs=stats.rankdata(m.t180_mean)
vals=[]
for perm in permutations(ys): vals.append(stats.pearsonr(xs,perm).statistic)
vals=np.array(vals); pexact=np.mean(np.abs(vals)>=abs(rho)-1e-12)
print('\ntradeoff rho',rho,'scipy p',stats.spearmanr(m.t180_mean,m.E_mJ).pvalue,'exact p',pexact)
print('tradeoff rho leave-one-out')
for i,row in m.iterrows():
 x=m.drop(i); print(row.specimen,stats.spearmanr(x.t180_mean,x.E_mJ).statistic)
# range
rng=df.t180_mean.max()-df.t180_mean.min()
print('\nrange',rng,'relative min %',100*rng/df.t180_mean.min(),'relative max %',100*rng/df.t180_mean.max(),'relative mean %',100*rng/df.t180_mean.mean())
# CI Pearson LOOCV
for rr in [.70,-.12]:
 ci=np.tanh(np.arctanh(rr)+np.array([-1,1])*stats.norm.ppf(.975)/np.sqrt(7-3))
 t=rr*np.sqrt((7-2)/(1-rr**2)); pp=2*stats.t.sf(abs(t),5)
 print('loocv r',rr,'CI',ci,'p',pp)
# design correlation matrix and univariate correlations to t180 and mass
cols=['H_mm','R_mm','cable_d_mm','strut_d_mm','twist_deg','mass_g','t180_mean']
print('\ncorrelation matrix\n',m[cols].corr().round(3).to_string())
# attenuation versus unity z using per-drop SEM and article noise .72% separately
for _,q in df.iterrows():
 sem=q.t180_sd/np.sqrt(q.n_valid)
 z=(q.t180_mean-1)/sem
 art_sd=.0072*q.t180_mean
 za=(q.t180_mean-1)/art_sd
 print(q.specimen,'mean',q.t180_mean,'perdrop SEM',sem,'z_sem',z,'z_articlefloor',za)


mapped n 7
specimen spec  mass_g  t180_mean  t180_sd   cv_pct      E_mJ
  6lhxfy   01   18.50   0.893078 0.004155 0.465295 13.928456
  6nheas   05   21.73   0.997008 0.003208 0.321714 13.070360
  9hhbkp   00   21.62   1.018336 0.001725 0.169399  6.947425
  autv5r   02   22.04   1.040430 0.003486 0.335059  8.830942
  bag26v   08   21.42   1.061620 0.005120 0.482290  7.714511
  bpx68c   S0   20.23   1.011072 0.002360 0.233429  6.180443
  nvxsrv   04   20.66   1.027549 0.004379 0.426184  8.202048

mass Pearson 0.8291263808685769 0.021095146051920175 Fisher95 [0.20251409 0.97402341]
mass r leave-one-out
6lhxfy 0.1899514290800765
6nheas 0.8988822617014381
9hhbkp 0.8379993109691679
autv5r 0.8182898921965868
bag26v 0.8456546647406413
bpx68c 0.8615686155718885
nvxsrv 0.8573779736704329



tradeoff rho -0.39285714285714296 scipy p 0.3833168704269727 exact p 0.3956349206349206
tradeoff rho leave-one-out
6lhxfy -0.028571428571428574
6nheas -0.028571428571428574
9hhbkp -0.6
autv5r -0.48571428571428577
bag26v -0.4285714285714286
bpx68c -0.6
nvxsrv -0.48571428571428577

range 0.1685419860411974 relative min % 18.872038734646953 relative max % 15.875927539843252 relative mean % 16.792092258133884
loocv r 0.7 CI [-0.11220696  0.95148936] p 0.07991669030889928
loocv r -0.12 CI [-0.80070117  0.69594891] p 0.7977452711162913

correlation matrix
              H_mm   R_mm  cable_d_mm  strut_d_mm  twist_deg  mass_g  t180_mean
H_mm        1.000 -0.092      -0.116      -0.490     -0.176   0.400      0.748
R_mm       -0.092  1.000       0.165      -0.375     -0.740   0.455      0.062
cable_d_mm -0.116  0.165       1.000      -0.692     -0.618   0.781      0.530
strut_d_mm -0.490 -0.375      -0.692       1.000      0.810  -0.914     -0.892
twist_deg  -0.176 -0.740      -0.618       0.81

In [2]:
from scipy import stats
import numpy as np, math
# Exact Spearman tail for n=7, rho=-13/14 likely
n=7; obs=13/14
base=np.arange(1,n+1)
rhos=[]
for perm in permutations(base):
    rhos.append(stats.pearsonr(base,perm).statistic)
rhos=np.array(rhos)
print('Exact two-sided p for |rho| >= 0.93:',np.mean(np.abs(rhos)>=obs-1e-12), 'count',np.sum(np.abs(rhos)>=obs-1e-12))
# Fisher rough CI shown for context despite Spearman
z=np.arctanh(-.93); se=1/np.sqrt(4); print('Fisher approximate CI rho',np.tanh(z+np.array([-1,1])*1.959964*se))
# normal-theory uncertainty in SD/CV estimated from only 5 articles
cv=.72; dfree=4
lo=cv*np.sqrt(dfree/stats.chi2.ppf(.975,dfree)); hi=cv*np.sqrt(dfree/stats.chi2.ppf(.025,dfree))
print('Normal-theory 95% CI for underlying article SD/CV from n=5:',lo,hi)
# current SD CV and SEM CV ranges all tested
cv=100*df.t180_sd/df.t180_mean
semcv=cv/np.sqrt(df.n_valid)
print('within drop CV range',cv.min(),cv.max())
print('per-session SEM CV range',semcv.min(),semcv.max())
print('article floor/perdrop SEM ratios',(.72/semcv).min(),(.72/semcv).max())
# normal approx interval around article response floor, for best article
mu=df.loc[df.specimen=='6lhxfy','t180_mean'].iloc[0]
print('best article ±1.96*0.72%:',mu + np.array([-1,1])*1.96*.0072*mu)


Exact two-sided p for |rho| >= 0.93: 0.006746031746031746 count 34
Fisher approximate CI rho [-0.98983386 -0.59048347]
Normal-theory 95% CI for underlying article SD/CV from n=5: 0.4313758601765814 2.0689600565363175
within drop CV range 0.1693993559981213 0.4822902636326832
per-session SEM CV range 0.01685586592157523 0.04798967487907
article floor/perdrop SEM ratios 15.00322729450325 42.71510009333972
best article ±1.96*0.72%: [0.88047467 0.9056809 ]
